In [ ]:
import pandas as pd

In [ ]:
item = pd.read_csv("C:/Users/mayan/Downloads/archive/olist_order_items_dataset.csv")
product = pd.read_csv("C:/Users/mayan/Downloads/archive/olist_products_dataset.csv")
translation = pd.read_csv("C:/Users/mayan/Downloads/archive/product_category_name_translation.csv")

In [ ]:
#item.head()
item.isnull().sum()

In [ ]:
product.isnull().sum()

In [ ]:
translation.isnull().sum()

In [ ]:
df = pd.merge(item , product , on='product_id' , how='left')
df = pd.merge(df , translation , on='product_category_name' , how='left')
df.drop('product_category_name',axis=1,inplace=True)

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df['product_category_name_english']=df['product_category_name_english'].fillna('other')
cols_to_fix = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in cols_to_fix:
    df[col] = df[col].fillna(0)
    
df.isnull().sum()

In [ ]:
# Load the orders dataset (this has the customer_id we need)
orders = pd.read_csv("C:/Users/mayan/Downloads/archive/olist_orders_dataset.csv")

# Merge it with your current 'df' using 'order_id' as the bridge
df = pd.merge(df, orders[['order_id', 'customer_id']], on='order_id', how='left')

In [ ]:
# Calculate Total Revenue by Category
revenue_by_cat = df.groupby('product_category_name_english')['price'].sum().sort_values(ascending=False).head(10)

# Calculate Average Order Value (AOV) by Category
aov_by_cat = df.groupby('product_category_name_english')['price'].mean().sort_values(ascending=False).head(10)

print("--- TOP 10 REVENUE CATEGORIES ---")
print(revenue_by_cat)
print("\n--- TOP 10 HIGHEST VALUE CATEGORIES (AOV) ---")
print(aov_by_cat)

In [ ]:
df.to_csv('cleaned_olist_data.csv', index=False)
print("Data exported! Ready for the Dashboard.")

In [ ]:
from sqlalchemy import create_engine
username = "root"
password = "7410"
host = "127.0.0.1"
port = "3306"
database = "olist_database"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

table_name = "olist_data"
df.to_sql(table_name, engine, if_exists="replace",index=False)

pd.read_sql("SELECT * FROM olive_dataset LIMIT 5;",engine)